In [ ]:
import polars as pl
from pathlib import Path
import shutil
import dotenv
from Bio import Entrez

In [ ]:
dotenv.load_dotenv()

email = dotenv.get_key(dotenv.find_dotenv(), "ENTREZ_EMAIL")
if email is None:
    raise ValueError("ENTREZ_EMAIL not found in .env file or environment variables.")
Entrez.email = email
print(f"Entrez email set to: {email}")

api_key = dotenv.get_key(dotenv.find_dotenv(), "ENTREZ_API_KEY")
if api_key is None:
    print("ENTREZ_API_KEY not found in .env file, proceeding without it.")
else:
    print("Entrez API key set.")
    Entrez.api_key = api_key

In [ ]:
# Assuming either all *_compressed.tar.zst files or all_groups_lean.tar.zst file is/are decompressed
CWD = Path().resolve()
BASE = CWD

print(f"Folder where groups (data) folder should be: {BASE}")

In [ ]:
source_path = Path("../igannot/sandbox/new_atlas").resolve()
source_path

In [ ]:
groups_map = {
    "plants_plt": ["solanaceae"],
    "plants_mit": ["solanaceae"],
}

In [ ]:
for g in groups_map.keys():
    dest_dir = BASE / g
    dest_dir.mkdir(parents=True, exist_ok=True)
    source_dir = source_path / g

    tsv_src = source_dir / f"{g}.tsv"
    tsv_dest = dest_dir / f"{g}.tsv"

    atlas_src = source_dir / f"{g}_summary_atlas.tsv"
    atlas_dest = dest_dir / f"{g}_summary_atlas.tsv"

    tree_src = source_dir / f"tree.nwk"
    tree_dest = dest_dir / f"tree.nwk"

    shutil.copy(atlas_src, atlas_dest)
    shutil.copy(tsv_src, tsv_dest)
    shutil.copy(tree_src, tree_dest)

In [ ]:
for g in groups_map.keys():
    tsv = pl.read_csv(BASE / g / f"{g}.tsv", separator="\t")

    taxids = tsv["ncbi_taxid"].unique().to_list()

In [ ]:
len(taxids)

In [ ]:
query_size = 250

taxid_map = {}

for window in range(0, len(taxids), query_size):
    taxid_window = taxids[window : window + query_size]
    ids = ",".join(map(str, taxid_window))
    print(
        f"Processing taxids {window} to {window + query_size} (total {len(taxid_window)})"
    )

    with Entrez.efetch(db="taxonomy", id=ids) as handle:
        records = Entrez.read(handle)

    for record in records:
        print(f"TaxId: {record['TaxId']}, Lineage: {record['Lineage']}")
        taxid_map[record["TaxId"]] = record["Lineage"]

In [ ]:
tsv = tsv.with_columns(
    pl.col("ncbi_taxid").replace_strict(taxid_map, default="Unknown").alias("lineage")
)
tsv.write_csv(BASE / g / f"{g}_raw_lineage.tsv", separator="\t")

## filtering subgroups

In [ ]:
keyword_map = {"solanaceae": ["Solanales", "Solanaceae"]}
prefix = r"(?:PLT|MIT)-"

for g, subgroups in groups_map.items():
    g_dir = BASE / g
    tsv = pl.read_csv(g_dir / f"{g}_raw_lineage.tsv", separator="\t")
    atlas = pl.read_csv(g_dir / f"{g}_summary_atlas.tsv", separator="\t")
    atlas = atlas.with_columns(
        pl.concat_str(
            pl.col("UP").str.replace_all(prefix, ""),
            pl.col("DOWN").str.replace_all(prefix, ""),
            pl.col("Polarity"),
            separator="_",
        ).alias("gene_pair")
    )
    # build once per group: split on ";", strip whitespace, drop empties (trailing ";")
    tsv = tsv.with_columns(
        pl.col("lineage")
        .str.split(";")
        .list.eval(pl.element().str.strip_chars())
        .list.eval(pl.element().filter(pl.element() != ""))
        .alias("lineage_taxa")
    )

    for subg in subgroups:
        if subg not in keyword_map:
            print(
                f"Warning: No keywords defined for subgroup '{subg}' in group '{g}'. Skipping."
            )
            continue

        keywords = keyword_map[subg]

        subg_tsv = tsv.filter(
            pl.any_horizontal(pl.col("lineage_taxa").list.contains(k) for k in keywords)
        ).drop("lineage", "lineage_taxa")

        before = subg_tsv.height
        subg_tsv = (
            subg_tsv.sort("Genome_length", descending=True)
            .unique(subset=["ncbi_taxid"], keep="first")
            .sort("AN")
        )
        print(
            f"Removed {before - subg_tsv.height} duplicates by ncbi_taxid for {subg} in {g}."
        )

        subg_dir = g_dir / subg
        subg_dir.mkdir(parents=True, exist_ok=True)

        subg_tsv.write_csv(subg_dir / f"{subg}.tsv", separator="\t")
        subg_atlas = atlas.filter(pl.col("AN").is_in(subg_tsv["AN"].to_list()))
        subg_atlas.write_csv(subg_dir / f"{subg}_summary_atlas.tsv", separator="\t")